# RNN Basics

In [94]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, Dense

In [95]:
sentences = [
    "The sun always shines brightest after the rain, bringing hope and new beginnings.",
    "A journey of a thousand miles begins with a single step, full of exciting discoveries.",
    "Innovation distinguishes between a leader and a follower, creating a brighter future.",
    "Believe you can and you're halfway there, with an unstoppable spirit.",
    "The only way to do great work is to love what you do, passionately and wholeheartedly.",
    "Success is not final, failure is not fatal: it is the courage to continue that counts, always moving forward.",
    "The future belongs to those who believe in the beauty of their dreams, a beautiful vision.",
    "In the middle of every difficulty lies opportunity, a chance to grow stronger.",
    "The best way to predict the future is to create it, with intention and effort.",
    "You are never too old to set another goal or to dream a new dream, for life is endless opportunity.",
    "Happiness is not something readymade. It comes from your own actions, making every moment count.",
    "What we achieve inwardly will change outer reality, transforming the world around us.",
    "Change your thoughts and you change your world, embracing positivity and growth.",
    "The best revenge is massive success, achieving goals beyond imagination.",
    "Spread love everywhere you go. Let no one ever come to you without leaving happier, radiating joy.",
    "The project encountered numerous unforeseen obstacles, leading to significant delays and cost overruns.",
    "Despite extensive efforts, the team was unable to resolve the critical software bug, causing frustration.",
    "The economic forecast indicates a sharp downturn, with widespread job losses expected across industries.",
    "A pervasive sense of unease settled over the town after the mysterious disappearance of several residents.",
    "The ancient prophecy spoke of an impending catastrophe that would engulf the world in darkness.",
    "The once vibrant ecosystem has been irrevocably damaged by pollution, threatening many species.",
    "He felt an overwhelming sense of despair as his dreams crumbled before his eyes, leaving him lost.",
    "The betrayal of his closest friend left an indelible scar, shattering his trust in humanity.",
    "The relentless storm raged for days, leaving a trail of destruction and widespread devastation.",
    "A deep-seated fear of failure paralyzed her, preventing her from pursuing her true passions.",
    "The bitter cold seeped into their bones, making every movement a painful struggle for survival.",
    "The oppressive silence of the empty house amplified her loneliness, a constant reminder of what was lost.",
    "The intricate web of lies he wove eventually unraveled, exposing his deception to everyone.",
    "She carried the heavy burden of guilt for years, unable to escape the shadows of her past actions.",
    "The bleak landscape stretched endlessly, a desolate reminder of a world ravaged by conflict."
]

lables = [1]*15 + [0]*15
lables = np.array(lables)

In [96]:
lables

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

In [97]:
vocab_size=2000

In [98]:
# for words out of vocabulary we are adding oov_token="<OOV>"
tokenize = Tokenizer(num_words=vocab_size, oov_token="<OOV>")

In [99]:
tokenize.fit_on_texts(sentences)

In [100]:
sequence = tokenize.texts_to_sequences(sentences)

In [101]:
max_length = max([len(seq) for seq in sequence])
max_length

20

In [102]:
X = pad_sequences(sequence, maxlen=max_length, padding="post")
y = lables

In [103]:
len(X)

30

In [104]:
X[0]

array([ 2, 51, 24, 52, 53, 25,  2, 54, 55, 56,  6, 26, 57,  0,  0,  0,  0,
        0,  0,  0], dtype=int32)

## Prepare Model

In [105]:
# we are creating embedding of 16 length

embed_dim = 16

In [106]:
# Total units you want to keep in hidden layers
rnn_units = 8

In [107]:
inp = Input(shape=(max_length,), dtype="int32", name="input")

x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True, name="embedding")(inp)

In [108]:
rnn = SimpleRNN(units=rnn_units, return_sequences=False, name="rnn")
x_last = rnn(x)
out = Dense(1, activation="sigmoid", name="output")(x_last)


In [109]:
model = Model(inputs=inp, outputs=out)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

In [110]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 20)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 20, 16)    │     32,000 │ input[0][0]       │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_4         │ (None, 20)        │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rnn (SimpleRNN)     │ (None, 8)         │        200 │ embedding[0][0],  │
│                     │                   │            │ not_equal_4[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 1)         │          9 │ rnn[0][0]         │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 32,209 (125.82 KB)

 Trainable params: 32,209 (125.82 KB)

 Non-trainable params: 0 (0.00 B)

## Train the model

In [111]:
model.fit(X, y, epochs=25, batch_size=8, verbose=1)

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.5667 - loss: 0.6923
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.8000 - loss: 0.6240
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9333 - loss: 0.5673
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.5145
Epoch 5/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.4626
Epoch 6/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.4126
Epoch 7/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.3663
Epoch 8/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.3225
Epoch 9/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.2840
Epoch 10/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 1.0000 - loss: 0.2489 
Epoch 11/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 1.0000 - loss: 0.2193
Epoch 12/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 1.0000 - loss: 0.1932
